In [2]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score
)

import joblib

In [3]:
train_df = pd.read_csv("fraud_train_features.csv")
test_df  = pd.read_csv("fraud_test_features.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (6536, 14)
Test shape: (1616, 14)


,num_detections,total_damage_area_ratio,avg_box_area_ratio,max_box_area_ratio,min_box_area_ratio,var_box_area_ratio,damage_density,class_1_count,class_2_count,class_3_count,class_4_count,class_entropy,mean_confidence,label
0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0,0,0.000000,0.000000,1
1,8,1.616940,0.202118,0.493783,0.005020,0.031520,0.202118,5,0,0,3,0.661563,0.737997,1
2,5,0.772816,0.154563,0.457177,0.000311,0.026961,0.154563,2,2,0,1,1.054920,0.647889,1
3,4,0.825834,0.206458,0.527934,0.045986,0.036008,0.206458,2,2,0,0,0.693147,0.591942,1
4,5,0.950058,0.190012,0.387256,0.019489,0.025478,0.190012,2,3,0,0,0.673012,0.605238,1


In [4]:
X_train = train_df.drop("label", axis=1)
y_train = train_df["label"]

X_test = test_df.drop("label", axis=1)
y_test = test_df["label"]

print("Fraud count in train:")
print(y_train.value_counts())

Fraud count in train:
label
0    6091
1     445
Name: count, dtype: int64


In [5]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [10]:
y_prob = rf_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.5245).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

roc = roc_auc_score(y_test, y_prob)
print("ROC-AUC:", round(roc, 4))

Confusion Matrix:
[[1521    2]
 [  23   70]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1523
           1       0.97      0.75      0.85        93

    accuracy                           0.98      1616
   macro avg       0.98      0.88      0.92      1616
weighted avg       0.98      0.98      0.98      1616

ROC-AUC: 0.9249


In [11]:
thresholds = np.linspace(0.1, 0.9, 50)

best_f1 = 0
best_threshold = 0

for t in thresholds:
    preds = (y_prob > t).astype(int)
    f1 = f1_score(y_test, preds)
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print("Best Threshold:", round(best_threshold, 4))
print("Best F1:", round(best_f1, 4))

Best Threshold: 0.5245
Best F1: 0.8485


In [12]:
final_pred = (y_prob > best_threshold).astype(int)

print("Final Confusion Matrix:")
print(confusion_matrix(y_test, final_pred))

print("\nFinal Classification Report:")
print(classification_report(y_test, final_pred))

precision = precision_score(y_test, final_pred)
recall = recall_score(y_test, final_pred)
f1 = f1_score(y_test, final_pred)
roc = roc_auc_score(y_test, y_prob)

print("\nFinal Metrics:")
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))
print("ROC-AUC:", round(roc, 4))

Final Confusion Matrix:
[[1521    2]
 [  23   70]]

Final Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1523
           1       0.97      0.75      0.85        93

    accuracy                           0.98      1616
   macro avg       0.98      0.88      0.92      1616
weighted avg       0.98      0.98      0.98      1616


Final Metrics:
Precision: 0.9722
Recall: 0.7527
F1 Score: 0.8485
ROC-AUC: 0.9249


In [13]:
joblib.dump(rf_model, "fraud_rf_model.pkl")

print("Fraud model saved as fraud_rf_model.pkl")

Fraud model saved as fraud_rf_model.pkl
